# **Neuroengineering project - Neural decoding** (Visualization code)

**Tutor**: Lorenzo Veronese, lorenzo.veronese@polimi.it

## Setup

In [2]:
from pathlib import Path

# Portable paths: run Jupyter from the repository root.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "supporting":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]
elif PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data directory:", PROJECT_ROOT / "data")
print("Output directory:", OUTPUT_DIR)


Mounted at /content/drive


In [3]:
# The portable setup cell selects the repository root.
os.chdir(PROJECT_ROOT)


In [4]:
# Install packages with requirements-supporting.txt before running.


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 78.6 MB/s eta 0:00:00


## Visualization

Script from: https://github.com/WeizmannVision/ssfmri2im/tree/master

In [5]:

from scipy.io import loadmat
import numpy as np
import pandas as pd
import sklearn.preprocessing
from sklearn import preprocessing


class kamitani_data_handler():
    """Generate batches for FMRI prediction
    frames_back - how many video frames to take before FMRI frame
    frames_forward - how many video frames to take after FMRI frame
    """

    def __init__(self, matlab_file ,test_img_csv = 'data/imageID_test.csv',train_img_csv = 'data/imageID_training.csv',voxel_spacing =3,log = 0 ):
        mat = loadmat(matlab_file)
        self.data = mat['dataSet'][:,3:]
        self.sample_meta = mat['dataSet'][:,:3]
        meta = mat['metaData']


        self.meta_keys = list(l[0] for l in meta[0][0][0][0])
        self.meta_desc = list(l[0] for l in meta[0][0][1][0])
        self.voxel_meta = np.nan_to_num(meta[0][0][2][:,3:])
        test_img_df = pd.read_csv(test_img_csv, header=None)
        train_img_df =pd.read_csv(train_img_csv, header=None)
        self.test_img_id = test_img_df[0].values
        self.train_img_id = train_img_df[0].values
        self.sample_type = {'train':1 , 'test':2 , 'test_imagine' : 3}
        self.voxel_spacing = voxel_spacing

        self.log = log

    def get_meta_field(self,field = 'DataType'):
        index = self.meta_keys.index(field)
        if(index <3): # 3 first keys are sample meta
            return self.sample_meta[:,index]
        else:
            return self.voxel_meta[index]


    def print_meta_desc(self):
        print(self.meta_desc)

    def get_labels(self, imag_data = 0,test_run_list = None):
        le = preprocessing.LabelEncoder()

        img_ids = self.get_meta_field('Label')
        type = self.get_meta_field('DataType')
        train = (type == self.sample_type['train'])
        test = (type == self.sample_type['test'])
        imag = (type == self.sample_type['test_imagine'])

        img_ids_train = img_ids[train]
        img_ids_test = img_ids[test]
        img_ids_imag = img_ids[imag]


        train_labels  = []
        test_labels  =  []
        imag_labels = []
        for id in img_ids_test:
            idx = (np.abs(id - self.test_img_id)).argmin()
            test_labels.append(idx)

        for id in img_ids_train:
            idx = (np.abs(id - self.train_img_id)).argmin()
            train_labels.append(idx)

        for id in img_ids_imag:
            idx = (np.abs(id - self.test_img_id)).argmin()
            imag_labels.append(idx)

        if (test_run_list is not None):
            run = self.get_meta_field('Run')
            test = (self.get_meta_field('DataType') == 2).astype(bool)
            run = run[test]

            select = np.in1d(run, test_run_list)
            test_labels = test_labels[select]

        #imag_labels = le.fit_transform(img_ids_imag)
        if(imag_data):
            return np.array(train_labels), np.array(test_labels), np.array(imag_labels)
        else:
            return np.array(train_labels),np.array(test_labels)





    def get_data(self,normalize =1 ,roi = 'ROI_VC',imag_data = 0,test_run_list = None):   # normalize 0-no, 1- per run , 2- train/test seperatly
        type = self.get_meta_field('DataType')
        train = (type == self.sample_type['train'])
        test = (type == self.sample_type['test'])
        test_imag = (type == self.sample_type['test_imagine'])
        test_all  = np.logical_or(test,test_imag)

        roi_select = self.get_meta_field(roi).astype(bool)
        data = self.data[:,roi_select]

        if(self.log ==1):
            data = np.log(1+np.abs(data))*np.sign(data)


        if(normalize==1):

            run = self.get_meta_field('Run').astype('int')-1
            num_runs = np.max(run)+1
            data_norm = np.zeros(data.shape)

            for r in range(num_runs):
                data_norm[r==run] = sklearn.preprocessing.scale(data[r==run])
            train_data = data_norm[train]
            test_data  = data_norm[test]
            test_all = data_norm[test_all]
            test_imag = data_norm[test_imag]

        else:
            train_data = data[train]
            test_data  =  data[test]
            if(normalize==2):
                train_data = sklearn.preprocessing.scale(train_data)
                test_data = sklearn.preprocessing.scale(test_data)


        if(self.log ==2):
            train_data = np.log(1+np.abs(train_data))*np.sign(train_data)
            test_data = np.log(1+np.abs(test_data))*np.sign(test_data)
            train_data = sklearn.preprocessing.scale(train_data)
            test_data = sklearn.preprocessing.scale(test_data)



        test_labels =  self.get_labels()[1]
        imag_labels = self.get_labels(1)[2]
        num_labels = max(test_labels)+1
        test_data_avg = np.zeros([num_labels,test_data.shape[1]])
        test_imag_avg = np.zeros([num_labels,test_data.shape[1]])

        if(test_run_list is not None):
            run = self.get_meta_field('Run')
            test = (self.get_meta_field('DataType') == 2).astype(bool)
            run = run[test]

            select = np.in1d(run, test_run_list)
            test_data = test_data[select,:]
            test_labels = test_labels[select]

        for i in range(num_labels):
            test_data_avg[i] = np.mean(test_data[test_labels==i],axis=0)
            test_imag_avg[i] = np.mean(test_imag[imag_labels == i], axis=0)
        if(imag_data):
            return train_data, test_data, test_data_avg,test_imag,test_imag_avg

        else:
            return train_data, test_data, test_data_avg

    def get_voxel_loc(self):
        x = self.get_meta_field('voxel_x')
        y = self.get_meta_field('voxel_y')
        z = self.get_meta_field('voxel_z')
        dim = [int(x.max() -x.min()+1),int(y.max() -y.min()+1), int(z.max() -z.min()+1)]
        return [x,y,z] , dim



ROI choice and data loading

In [6]:
import os
import math
from keras.callbacks import TensorBoard, LearningRateScheduler

handler = kamitani_data_handler(matlab_file = "data/Subject1.mat")
Y,Y_test,Y_test_avg = handler.get_data(roi = 'ROI_VC',imag_data = 0) #Change 'roi' string according to the ROI to mask on

In [7]:
# get voxel coordinates and volume dimensions
(vox_x, vox_y, vox_z), dims = handler.get_voxel_loc()

# take one sample's voxel values (e.g. first row)
sample_voxels = Y[0]

# initialize empty 3D volume
volume = np.zeros(dims)

# fill in voxel values at the correct locations
for val, x, y, z in zip(sample_voxels, vox_x, vox_y, vox_z):
    volume[int(x), int(y), int(z)] = val


Visualize. One activation map is superimposed on the T1 anatomical scan

In [8]:
import numpy as np
import plotly.graph_objects as go
from skimage import measure
import nibabel as nib

# --- Load subject T1-weighted anatomical scan ---
t1_img = nib.load("data/Subject1_T1wAligned.nii")
t1_data = t1_img.get_fdata()
template_shape = t1_data.shape

# --- Load Kamitani fMRI data ---
handler = kamitani_data_handler(matlab_file="data/Subject1.mat")
Y, Y_test, Y_test_avg = handler.get_data(roi='ROI_VC', imag_data=0)
x, y, z = [c.astype(int) for c in handler.get_voxel_loc()[0]]

# --- Normalize ROI coordinates to start at 0 ---
x = x - x.min()
y = y - y.min()
z = z - z.min()

# --- Scale ROI to fit in template (occipital lobe region) ---
roi_size = np.array([x.max(), y.max(), z.max()])
roi_center = roi_size / 2
template_center = np.array(template_shape) / 2  # center of T1
offset = template_center - roi_center
x_t = np.clip((x + offset[0]).astype(int), 0, template_shape[0]-1)
y_t = np.clip((y + offset[1]).astype(int), 0, template_shape[1]-1)
z_t = np.clip((z + offset[2]).astype(int), 0, template_shape[2]-1)

# --- Adjust ROI coordinates for translation ---
x_offset_move = -5   # move forward along x-axis
y_offset_move = -45  # move back along y-axis
z_offset_move = 0    # no change along z-axis

x_t_moved = np.clip(x_t + x_offset_move, 0, template_shape[0]-1)
y_t_moved = np.clip(y_t + y_offset_move, 0, template_shape[1]-1)
z_t_moved = z_t + z_offset_move  # unchanged

# --- Mirror blob along x-axis ---
x_t_flipped = template_shape[0] - 1 - x_t_moved

# --- Ensure lengths match ROI voxel values ---
num_voxels = Y_test_avg[0].shape[0]
x_t_flipped = x_t_flipped[:num_voxels]
y_t_moved = y_t_moved[:num_voxels]
z_t_moved = z_t_moved[:num_voxels]

# --- Create activation volume safely ---
volume = np.zeros(template_shape)
for i in range(num_voxels):
    xi = int(x_t_flipped[i])
    yi = int(y_t_moved[i])
    zi = int(z_t_moved[i])
    volume[xi, yi, zi] = Y_test_avg[0][i]

# --- Optional rotation if needed ---
# volume = np.rot90(volume, k=1, axes=(1, 2))  # rotate top
# volume = np.rot90(volume, k=1, axes=(0, 2))  # rotate right

# --- T1 surface ---
t1_threshold = np.percentile(t1_data, 70)
verts, faces, _, _ = measure.marching_cubes(t1_data, level=t1_threshold)
t1_mesh = go.Mesh3d(
    x=verts[:,0], y=verts[:,1], z=verts[:,2],
    i=faces[:,0], j=faces[:,1], k=faces[:,2],
    color='lightgray', opacity=0.3, name='Subject T1'
)

# --- Activation surface with per-vertex intensity ---
threshold = np.percentile(volume[volume>0], 10)
verts_act, faces_act, _, _ = measure.marching_cubes(volume, level=threshold)

# Map vertex intensities from the volume
# nearest neighbor interpolation from voxel values to vertices
xv = np.clip(verts_act[:,0].astype(int), 0, template_shape[0]-1)
yv = np.clip(verts_act[:,1].astype(int), 0, template_shape[1]-1)
zv = np.clip(verts_act[:,2].astype(int), 0, template_shape[2]-1)
intensity = volume[xv, yv, zv]

act_mesh = go.Mesh3d(
    x=verts_act[:,0], y=verts_act[:,1], z=verts_act[:,2],
    i=faces_act[:,0], j=faces_act[:,1], k=faces_act[:,2],
    intensity=intensity,  # this makes color reflect values
    colorscale='Reds',
    opacity=0.6,
    name='V1 Activation'
)

# --- Plot ---
fig = go.Figure(data=[t1_mesh, act_mesh])
fig.update_layout(
    scene=dict(
        xaxis_title='X (mm)', yaxis_title='Y (mm)', zaxis_title='Z (mm)',
        aspectmode='data'
    ),
    title="fMRI V1 Activations over Subject T1"
)
fig.show()


Output hidden; open in https://colab.research.google.com to view.